# EV Dataset Harmonization Pipeline

**MSc Data Science Dissertation**

Author: Tarek Noman

Purpose:

This notebook harmonizes the processed VED, BMW i3, and Argonne BMW i3 datasets into a unified machine learning dataset for predicting electric vehicle energy consumption.


## Stage 1 — Imports


In [1]:
# ==========================================================
# Stage 1 - Imports
# ==========================================================

from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Libraries imported successfully.")

Libraries imported successfully.


## Stage 2 — Dataset Paths

In [2]:
# ==========================================================
# Stage 2 - Dataset Paths
# ==========================================================

PROJECT_ROOT = Path(r"C:\Projects\DS7010_Dissertation")

VED_PATH = (
    PROJECT_ROOT
    / "01-Datasets"
    / "VED_BEV_ML_TABLE.csv"
)

BMW_PATH = (
    PROJECT_ROOT
    / "02-Processed"
    / "BMW"
    / "BMW_ML_TABLE.csv"
)

ARGONNE_PATH = (
    PROJECT_ROOT
    / "02-Processed"
    / "Argonne"
    / "ARGONNE_ML_TABLE.csv"
)

HARMONIZED_OUTPUT_DIR = (
    PROJECT_ROOT
    / "02-Processed"
    / "Harmonized"
)

HARMONIZED_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("VED file exists:", VED_PATH.exists())
print("BMW file exists:", BMW_PATH.exists())
print("Argonne file exists:", ARGONNE_PATH.exists())
print("Output folder:", HARMONIZED_OUTPUT_DIR)

VED file exists: True
BMW file exists: True
Argonne file exists: True
Output folder: C:\Projects\DS7010_Dissertation\02-Processed\Harmonized


In [3]:
datasets_dir = PROJECT_ROOT / "01-Datasets"

print("Files matching VED_BEV_ML_TABLE:")

matches = list(datasets_dir.glob("VED_BEV_ML_TABLE*"))

for path in matches:
    print(
        "Name:", path.name,
        "| Suffix:", path.suffix,
        "| Exists:", path.exists()
    )

Files matching VED_BEV_ML_TABLE:
Name: VED_BEV_ML_TABLE.csv | Suffix: .csv | Exists: True


In [4]:
print(VED_PATH)
print(VED_PATH.exists())


C:\Projects\DS7010_Dissertation\01-Datasets\VED_BEV_ML_TABLE.csv
True


In [5]:
from pathlib import Path

print("PROJECT_ROOT:")
print(PROJECT_ROOT)

print("\nResolved PROJECT_ROOT:")
print(PROJECT_ROOT.resolve())

print("\nContents of PROJECT_ROOT:")
for item in PROJECT_ROOT.iterdir():
    print(item.name)

print("\nContents of 01-Datasets:")
for item in (PROJECT_ROOT / "01-Datasets").iterdir():
    print(item.name)

PROJECT_ROOT:
C:\Projects\DS7010_Dissertation

Resolved PROJECT_ROOT:
C:\Projects\DS7010_Dissertation

Contents of PROJECT_ROOT:
.git
.gitignore
01-Datasets
02-Literature
02-Processed
03-Notebooks
04-Code
05-Figures
06-Results
07-Dissertation writting
08-Prsesntation
09-Research_Log
10-Backups

Contents of 01-Datasets:
01-VED
02-BMW-i3
03-Argonne_D3
04-Yuan 2024
Harmonised
VED_ALL.csv
VED_BEV_AUDIT_TABLE.csv
VED_BEV_ML_TABLE.csv
VED_BEV_ONLY.csv
VED_EV_ONLY.csv
VED_EV_SAMPLE_200K.csv


## Stage 3 — Load Processed Datasets

In [6]:
# ==========================================================
# Stage 3 - Load Processed Datasets
# ==========================================================

ved = pd.read_csv(VED_PATH)
bmw = pd.read_csv(BMW_PATH)
argonne = pd.read_csv(ARGONNE_PATH)

print("=" * 60)
print("DATASETS LOADED")
print("=" * 60)

print("\nVED shape:", ved.shape)
print("BMW shape:", bmw.shape)
print("Argonne shape:", argonne.shape)

DATASETS LOADED

VED shape: (488, 16)
BMW shape: (68, 16)
Argonne shape: (19, 17)


## Stage 4 — Compare Dataset Schemas

In [7]:
# ==========================================================
# Stage 4 - Compare Dataset Schemas
# ==========================================================

print("=" * 60)
print("VED COLUMNS")
print("=" * 60)
for col in ved.columns:
    print("-", col)

print("\n" + "=" * 60)
print("BMW COLUMNS")
print("=" * 60)
for col in bmw.columns:
    print("-", col)

print("\n" + "=" * 60)
print("ARGONNE COLUMNS")
print("=" * 60)
for col in argonne.columns:
    print("-", col)

VED COLUMNS
- Segment_ID
- VehId
- Trip
- Duration_s
- Mean_Speed_kmh
- Max_Speed_kmh
- Std_Speed_kmh
- Mean_Acceleration_mps2
- Mean_Positive_Acceleration_mps2
- Max_Acceleration_mps2
- Min_Acceleration_mps2
- Mean_Temp_C
- Mean_Heater_W
- Idling_Proportion
- RPA_mps2
- Energy_Consumption_kWh_100km

BMW COLUMNS
- Trip_ID
- Mean_Speed_kmh
- Max_Speed_kmh
- Std_Speed_kmh
- Mean_Acceleration_mps2
- Max_Acceleration_mps2
- Min_Acceleration_mps2
- Mean_Positive_Acceleration_mps2
- Idling_Proportion
- Start_SOC
- Mean_Ambient_Temp_C
- Mean_Battery_Temp_C
- Mean_Heating_kW
- Mean_AirCon_kW
- Duration_s
- Energy_Consumption_kWh_100km

ARGONNE COLUMNS
- Trip_ID
- Mean_Speed_kmh
- Max_Speed_kmh
- Std_Speed_kmh
- Mean_Acceleration_mps2
- Max_Acceleration_mps2
- Min_Acceleration_mps2
- Mean_Positive_Acceleration_mps2
- Idling_Proportion
- Start_SOC
- Mean_Test_Cell_Temp_C
- Mean_Test_Cell_RH
- Mean_Motor_RPM
- Mean_Tractive_Effort_N
- Mean_Accelerator_Position
- Duration_s
- Energy_Consumption_kW

## Stage 5 — Harmonize Features

In [8]:
# ==========================================================
# Stage 5 - Harmonize Features
# ==========================================================

COMMON_COLUMNS = [
    "Mean_Speed_kmh",
    "Max_Speed_kmh",
    "Std_Speed_kmh",
    "Mean_Acceleration_mps2",
    "Max_Acceleration_mps2",
    "Min_Acceleration_mps2",
    "Mean_Positive_Acceleration_mps2",
    "Idling_Proportion",
    "Duration_s",
    "Energy_Consumption_kWh_100km"
]

# Keep only common columns
ved_common = ved[COMMON_COLUMNS].copy()
bmw_common = bmw[COMMON_COLUMNS].copy()
argonne_common = argonne[COMMON_COLUMNS].copy()

# Add dataset labels
ved_common["Dataset"] = "VED"
bmw_common["Dataset"] = "BMW"
argonne_common["Dataset"] = "ARGONNE"

print("VED:", ved_common.shape)
print("BMW:", bmw_common.shape)
print("Argonne:", argonne_common.shape)

VED: (488, 11)
BMW: (68, 11)
Argonne: (19, 11)


## Stage 6 — Merge Datasets

In [9]:
# ==========================================================
# Stage 6 - Merge Datasets
# ==========================================================

master_dataset = pd.concat(
    [
        ved_common,
        bmw_common,
        argonne_common
    ],
    ignore_index=True
)

print("=" * 60)
print("MASTER DATASET")
print("=" * 60)

print("\nShape:")
print(master_dataset.shape)

print("\nRows per dataset:")

print(
    master_dataset["Dataset"]
    .value_counts()
)

display(master_dataset.head())

MASTER DATASET

Shape:
(575, 11)

Rows per dataset:
Dataset
VED        488
BMW         68
ARGONNE     19
Name: count, dtype: int64


,Mean_Speed_kmh,Max_Speed_kmh,Std_Speed_kmh,Mean_Acceleration_mps2,Max_Acceleration_mps2,Min_Acceleration_mps2,Mean_Positive_Acceleration_mps2,Idling_Proportion,Duration_s,Energy_Consumption_kWh_100km,Dataset
0,35.550077,54.629997,16.473919,-0.013622,9.333333,-5.874999,0.328515,0.062016,106.2,13.760491,VED
1,22.925093,53.049999,15.883698,-0.010552,7.638889,-7.986111,0.355230,0.099251,460.4,28.697582,VED
2,19.378746,41.829998,13.809472,-0.028062,8.611111,-9.638889,0.320445,0.160763,370.3,35.971756,VED
3,26.365563,49.090000,17.445833,0.028884,6.425925,-7.152777,0.249872,0.151408,313.7,22.824763,VED
4,24.035848,58.739998,16.030381,0.012083,9.333335,-6.750000,0.416068,0.111607,382.2,27.585331,VED


## Stage 7 — Final Audit

In [10]:
# ==========================================================
# Stage 7 - Final Audit
# ==========================================================

print("="*70)
print("MASTER DATASET AUDIT")
print("="*70)

print("\nShape:")
print(master_dataset.shape)

print("\nMissing values:")
display(master_dataset.isna().sum())

print("\nDuplicate rows:")
print(master_dataset.duplicated().sum())

print("\nDataset distribution:")
display(
    master_dataset["Dataset"].value_counts()
)

print("\nTarget statistics by dataset:")
display(
    master_dataset.groupby("Dataset")[
        "Energy_Consumption_kWh_100km"
    ].describe()
)

print("\nOverall statistics:")
display(master_dataset.describe())

MASTER DATASET AUDIT

Shape:
(575, 11)

Missing values:


Mean_Speed_kmh                     0
Max_Speed_kmh                      0
Std_Speed_kmh                      0
Mean_Acceleration_mps2             0
Max_Acceleration_mps2              0
Min_Acceleration_mps2              0
Mean_Positive_Acceleration_mps2    0
Idling_Proportion                  0
Duration_s                         0
Energy_Consumption_kWh_100km       0
Dataset                            0
dtype: int64


Duplicate rows:
0

Dataset distribution:


Dataset
VED        488
BMW         68
ARGONNE     19
Name: count, dtype: int64


Target statistics by dataset:


,count,mean,std,min,25%,50%,75%,max
Dataset,,,,,,,,
ARGONNE,19.0,17.332149,7.164990,4.004945,13.119734,15.813266,20.010499,37.690272
BMW,68.0,16.474035,3.460082,9.576267,13.729263,16.514365,19.115315,23.301433
VED,488.0,16.058173,7.479036,2.846425,11.655858,14.579965,18.570516,79.163078



Overall statistics:


,Mean_Speed_kmh,Max_Speed_kmh,Std_Speed_kmh,Mean_Acceleration_mps2,Max_Acceleration_mps2,Min_Acceleration_mps2,Mean_Positive_Acceleration_mps2,Idling_Proportion,Duration_s,Energy_Consumption_kWh_100km
count,575.000000,575.000000,575.000000,575.000000,575.000000,575.000000,575.000000,575.000000,575.000000,575.000000
mean,41.581699,78.431892,21.987999,0.004857,7.380085,-7.372853,0.222182,0.105965,751.367913,16.149450
std,12.859014,22.909697,7.160963,0.039287,2.437758,2.402343,0.062090,0.086727,787.874509,7.108521
min,4.093332,27.917221,1.890054,-0.353214,1.452876,-10.000000,0.052985,0.000000,60.400000,2.846425
25%,33.040256,65.489998,17.696307,-0.012826,5.541669,-9.254632,0.184299,0.051356,282.850000,11.898583
50%,40.265499,72.680000,20.846451,0.004578,8.305555,-8.249998,0.216685,0.092421,522.400000,14.973896
75%,48.170826,82.574997,24.231111,0.023786,9.416666,-6.277778,0.252292,0.136087,929.000000,18.827392
max,95.699948,152.260000,48.979649,0.196149,10.000000,-1.381946,0.695104,0.792473,5337.995000,79.163078


In [11]:
print("="*70)
print("Correlation Matrix")
print("="*70)

corr = master_dataset.drop(
    columns="Dataset"
).corr(numeric_only=True)

display(corr)

Correlation Matrix


,Mean_Speed_kmh,Max_Speed_kmh,Std_Speed_kmh,Mean_Acceleration_mps2,Max_Acceleration_mps2,Min_Acceleration_mps2,Mean_Positive_Acceleration_mps2,Idling_Proportion,Duration_s,Energy_Consumption_kWh_100km
Mean_Speed_kmh,1.000000,0.605265,0.439185,-0.052668,-0.302388,0.250046,-0.169120,-0.495833,-0.102176,-0.281052
Max_Speed_kmh,0.605265,1.000000,0.855757,0.004085,-0.462904,0.446656,0.103514,0.024902,0.468354,-0.044893
Std_Speed_kmh,0.439185,0.855757,1.000000,-0.029473,-0.354187,0.330819,0.119367,0.264752,0.418906,0.045860
Mean_Acceleration_mps2,-0.052668,0.004085,-0.029473,1.000000,0.237531,0.108461,0.311225,-0.008994,0.043118,0.176891
Max_Acceleration_mps2,-0.302388,-0.462904,-0.354187,0.237531,1.000000,-0.702827,0.025968,-0.075899,-0.215785,0.069602
Min_Acceleration_mps2,0.250046,0.446656,0.330819,0.108461,-0.702827,1.000000,0.104613,0.076225,0.236164,-0.012467
Mean_Positive_Acceleration_mps2,-0.169120,0.103514,0.119367,0.311225,0.025968,0.104613,1.000000,0.163314,0.179322,0.096615
Idling_Proportion,-0.495833,0.024902,0.264752,-0.008994,-0.075899,0.076225,0.163314,1.000000,0.414525,0.410105
Duration_s,-0.102176,0.468354,0.418906,0.043118,-0.215785,0.236164,0.179322,0.414525,1.000000,0.064893
Energy_Consumption_kWh_100km,-0.281052,-0.044893,0.045860,0.176891,0.069602,-0.012467,0.096615,0.410105,0.064893,1.000000


## Stage 8 — Export Master Dataset

In [12]:
# ==========================================================
# Stage 8 - Export Master Dataset
# ==========================================================

master_audit = master_dataset.copy()

master_ml = master_dataset.copy()

audit_path = (
    HARMONIZED_OUTPUT_DIR
    / "MASTER_EV_AUDIT.csv"
)

ml_path = (
    HARMONIZED_OUTPUT_DIR
    / "MASTER_EV_ML.csv"
)

master_audit.to_csv(
    audit_path,
    index=False
)

master_ml.to_csv(
    ml_path,
    index=False
)

print("=" * 70)
print("MASTER DATASET EXPORTED")
print("=" * 70)

print("\nAudit:")
print(audit_path)

print("\nML:")
print(ml_path)

print("\nAudit shape:", master_audit.shape)
print("ML shape:", master_ml.shape)

MASTER DATASET EXPORTED

Audit:
C:\Projects\DS7010_Dissertation\02-Processed\Harmonized\MASTER_EV_AUDIT.csv

ML:
C:\Projects\DS7010_Dissertation\02-Processed\Harmonized\MASTER_EV_ML.csv

Audit shape: (575, 11)
ML shape: (575, 11)


## "EXTREME TARGET AUDI

In [13]:
print("EXTREME TARGET AUDIT")
print("=" * 60)

for dataset in ["VED", "BMW", "ARGONNE"]:
    temp = master_dataset[
        master_dataset["Dataset"] == dataset
    ].sort_values(
        "Energy_Consumption_kWh_100km",
        ascending=False
    )

    print(f"\n{dataset} - highest 10 targets:")
    display(
        temp[
            [
                "Mean_Speed_kmh",
                "Max_Speed_kmh",
                "Std_Speed_kmh",
                "Idling_Proportion",
                "Duration_s",
                "Energy_Consumption_kWh_100km"
            ]
        ].head(10)
    )

EXTREME TARGET AUDIT

VED - highest 10 targets:


,Mean_Speed_kmh,Max_Speed_kmh,Std_Speed_kmh,Idling_Proportion,Duration_s,Energy_Consumption_kWh_100km
29,11.625928,29.969999,9.445854,0.297634,522.4,79.163078
30,12.185868,71.049995,17.174703,0.527640,2192.2,65.109691
23,16.471678,50.360001,17.043051,0.383051,388.6,54.308139
15,36.466653,67.299995,24.525640,0.212766,578.9,44.040044
48,23.384991,58.989998,16.768524,0.154270,959.8,41.561471
22,30.221453,77.320000,20.645925,0.143841,682.4,39.469066
448,24.900694,36.369999,7.015459,0.000000,180.6,38.792530
47,31.193737,73.529999,18.794220,0.115440,713.9,37.314101
462,24.205088,47.230000,13.819800,0.113739,1029.5,37.258791
2,19.378746,41.829998,13.809472,0.160763,370.3,35.971756



BMW - highest 10 targets:


,Mean_Speed_kmh,Max_Speed_kmh,Std_Speed_kmh,Idling_Proportion,Duration_s,Energy_Consumption_kWh_100km
521,42.284887,135.12,27.715166,0.063303,1611.2,23.301433
540,60.485600,142.23,48.979649,0.075182,1113.2,21.783572
549,64.422194,148.61,46.613140,0.138603,795.7,21.739404
546,58.848103,151.23,46.800438,0.191926,968.5,21.225910
547,58.093566,119.74,39.769862,0.103811,920.8,21.104824
535,51.310532,122.31,37.912941,0.131320,1109.4,21.052643
548,50.021854,114.69,38.361655,0.082870,1096.8,21.015502
554,44.355806,106.84,28.627880,0.097439,1417.2,21.002172
532,38.007697,141.05,28.201765,0.137628,1822.2,20.925972
527,85.118739,137.32,43.572104,0.068107,2023.2,20.891196



ARGONNE - highest 10 targets:


,Mean_Speed_kmh,Max_Speed_kmh,Std_Speed_kmh,Idling_Proportion,Duration_s,Energy_Consumption_kWh_100km
557,65.573020,130.531958,36.940607,0.038538,549.999,37.690272
558,72.226525,133.272664,41.516863,0.137421,811.999,26.538541
573,39.807207,129.117348,35.092845,0.269870,4829.995,25.256664
574,37.127759,129.588885,44.681539,0.499515,1443.499,20.946449
571,39.851558,129.670961,35.120507,0.268773,4829.995,20.760446
570,4.093332,27.917221,8.818705,0.792473,92.900,19.260553
572,41.062159,129.199425,37.716696,0.307495,3112.797,18.353847
568,39.849533,129.814192,35.119333,0.269580,4829.995,17.695855
563,95.699948,106.913284,26.534815,0.040856,509.000,15.869896
567,92.364641,108.070400,32.336608,0.088539,678.699,15.813266


# Temperature Availability Audit

In [14]:
# ============================================================
# Temperature Availability Audit
# ============================================================

dataframes_to_check = {
    "VED": ved,
    "BMW": bmw,
    "ARGONNE": argonne
}

temperature_keywords = [
    "temp",
    "temperature",
    "ambient",
    "oat",
    "outside"
]

for dataset_name, df in dataframes_to_check.items():

    print("=" * 70)
    print(dataset_name)
    print("=" * 70)

    matching_columns = [
        col for col in df.columns
        if any(
            keyword in col.lower()
            for keyword in temperature_keywords
        )
    ]

    if matching_columns:
        print("Possible temperature columns:")
        for col in matching_columns:
            print(" -", col)
    else:
        print("No obvious temperature column found.")

VED
Possible temperature columns:
 - Mean_Temp_C
BMW
Possible temperature columns:
 - Mean_Ambient_Temp_C
 - Mean_Battery_Temp_C
ARGONNE
Possible temperature columns:
 - Mean_Test_Cell_Temp_C


# Temperature Audit - Values, Missingness and Ranges

In [15]:
# ============================================================
# Temperature Audit - Values, Missingness and Ranges
# ============================================================

temperature_columns = {
    "VED": (ved, "Mean_Temp_C"),
    "BMW": (bmw, "Mean_Ambient_Temp_C"),
    "ARGONNE": (argonne, "Mean_Test_Cell_Temp_C")
}

for dataset_name, (df, temp_col) in temperature_columns.items():

    print("=" * 70)
    print(dataset_name)
    print("Column:", temp_col)
    print("=" * 70)

    print("Rows:", len(df))
    print("Missing:", df[temp_col].isna().sum())

    print("\nStatistics:")
    print(df[temp_col].describe())

    print("\nLowest 5:")
    print(df[temp_col].nsmallest(5).values)

    print("\nHighest 5:")
    print(df[temp_col].nlargest(5).values)

    print()

VED
Column: Mean_Temp_C
Rows: 488
Missing: 0

Statistics:
count    488.000000
mean      12.348002
std       11.502825
min      -14.452944
25%        3.074148
50%       11.521070
75%       22.199994
max       35.529172
Name: Mean_Temp_C, dtype: float64

Lowest 5:
[-14.45294414 -13.15320122 -12.65460526 -11.88179036 -10.72773973]

Highest 5:
[35.52917232 35.15159574 33.90497449 33.37873134 32.95170455]

BMW
Column: Mean_Ambient_Temp_C
Rows: 68
Missing: 0

Statistics:
count    68.000000
mean     13.213831
std       9.674610
min      -3.271403
25%       4.925246
50%       9.089179
75%      21.916412
max      31.127573
Name: Mean_Ambient_Temp_C, dtype: float64

Lowest 5:
[-3.27140267 -0.86986     1.08874145  2.19325836  2.34831128]

Highest 5:
[31.12757325 30.76997225 30.18323432 29.1681264  29.06001196]

ARGONNE
Column: Mean_Test_Cell_Temp_C
Rows: 19
Missing: 0

Statistics:
count    19.000000
mean     16.045639
std      17.966502
min     -16.107683
25%       1.591465
50%      23.434814
75%

# Physical Plausibility Audit
# Acceleration + Energy Consumption

In [16]:
# ============================================================
# Physical Plausibility Audit
# Acceleration + Energy Consumption
# ============================================================

datasets = {
    "VED": ved,
    "BMW": bmw,
    "ARGONNE": argonne
}

audit_columns = [
    "Mean_Acceleration_mps2",
    "Max_Acceleration_mps2",
    "Min_Acceleration_mps2",
    "Mean_Positive_Acceleration_mps2",
    "Energy_Consumption_kWh_100km"
]

for dataset_name, df in datasets.items():

    print("=" * 75)
    print(dataset_name)
    print("=" * 75)

    for col in audit_columns:

        if col not in df.columns:
            print(f"\n{col}: NOT FOUND")
            continue

        s = df[col].dropna()

        print(f"\n{col}")
        print("-" * 50)

        print(f"Count : {len(s)}")
        print(f"Mean  : {s.mean():.4f}")
        print(f"Std   : {s.std():.4f}")
        print(f"Min   : {s.min():.4f}")
        print(f"1%    : {s.quantile(0.01):.4f}")
        print(f"5%    : {s.quantile(0.05):.4f}")
        print(f"50%   : {s.median():.4f}")
        print(f"95%   : {s.quantile(0.95):.4f}")
        print(f"99%   : {s.quantile(0.99):.4f}")
        print(f"Max   : {s.max():.4f}")

    print()

VED

Mean_Acceleration_mps2
--------------------------------------------------
Count : 488
Mean  : 0.0050
Std   : 0.0408
Min   : -0.3532
1%    : -0.0968
5%    : -0.0579
50%   : 0.0059
95%   : 0.0613
99%   : 0.1016
Max   : 0.1961

Max_Acceleration_mps2
--------------------------------------------------
Count : 488
Mean  : 8.1442
Std   : 1.7520
Min   : 1.6111
1%    : 3.4910
5%    : 4.5926
50%   : 8.8056
95%   : 9.9167
99%   : 10.0000
Max   : 10.0000

Min_Acceleration_mps2
--------------------------------------------------
Count : 488
Mean  : -8.1374
Std   : 1.6568
Min   : -10.0000
1%    : -9.9758
5%    : -9.9028
50%   : -8.5556
95%   : -4.7797
99%   : -2.4152
Max   : -1.3819

Mean_Positive_Acceleration_mps2
--------------------------------------------------
Count : 488
Mean  : 0.2165
Std   : 0.0499
Min   : 0.0530
1%    : 0.1056
5%    : 0.1447
50%   : 0.2140
95%   : 0.3025
99%   : 0.3517
Max   : 0.4161

Energy_Consumption_kWh_100km
--------------------------------------------------
Count 

# Acceleration Boundary + Extreme Target Audit

In [17]:
# ============================================================
# Acceleration Boundary + Extreme Target Audit
# ============================================================

datasets = {
    "VED": ved,
    "BMW": bmw,
    "ARGONNE": argonne
}

for name, df in datasets.items():

    print("=" * 70)
    print(name)
    print("=" * 70)

    # Acceleration boundary checks
    print("\nAcceleration checks:")

    print("Max acceleration >= 10:",
          (df["Max_Acceleration_mps2"] >= 10).sum())

    print("Max acceleration >= 8:",
          (df["Max_Acceleration_mps2"] >= 8).sum())

    print("Min acceleration <= -10:",
          (df["Min_Acceleration_mps2"] <= -10).sum())

    print("Min acceleration <= -8:",
          (df["Min_Acceleration_mps2"] <= -8).sum())

    # Target checks
    target = df["Energy_Consumption_kWh_100km"]

    print("\nEnergy target checks:")

    for threshold in [30, 40, 50, 60, 70]:
        print(
            f"Energy > {threshold}:",
            (target > threshold).sum()
        )

    print("\nTop 10 energy-consumption observations:")

    columns_to_show = [
        "Mean_Speed_kmh",
        "Max_Speed_kmh",
        "Mean_Acceleration_mps2",
        "Max_Acceleration_mps2",
        "Min_Acceleration_mps2",
        "Mean_Positive_Acceleration_mps2",
        "Idling_Proportion",
        "Duration_s",
        "Energy_Consumption_kWh_100km"
    ]

    # Include temperature if available
    temp_candidates = [
        "Mean_Temp_C",
        "Mean_Ambient_Temp_C",
        "Mean_Test_Cell_Temp_C"
    ]

    for col in temp_candidates:
        if col in df.columns:
            columns_to_show.append(col)

    display(
        df.nlargest(
            10,
            "Energy_Consumption_kWh_100km"
        )[columns_to_show]
    )

VED

Acceleration checks:
Max acceleration >= 10: 0
Max acceleration >= 8: 306
Min acceleration <= -10: 0
Min acceleration <= -8: 314

Energy target checks:
Energy > 30: 19
Energy > 40: 5
Energy > 50: 3
Energy > 60: 2
Energy > 70: 1

Top 10 energy-consumption observations:


,Mean_Speed_kmh,Max_Speed_kmh,Mean_Acceleration_mps2,Max_Acceleration_mps2,Min_Acceleration_mps2,Mean_Positive_Acceleration_mps2,Idling_Proportion,Duration_s,Energy_Consumption_kWh_100km,Mean_Temp_C
29,11.625928,29.969999,0.012040,6.222222,-6.777774,0.166863,0.297634,522.4,79.163078,-7.500000
30,12.185868,71.049995,0.007261,9.777779,-9.750001,0.150808,0.527640,2192.2,65.109691,-6.510276
23,16.471678,50.360001,-0.033974,8.124999,-8.314815,0.193065,0.383051,388.6,54.308139,-4.027966
15,36.466653,67.299995,0.063072,7.888889,-9.472222,0.232789,0.212766,578.9,44.040044,1.240812
48,23.384991,58.989998,-0.011981,9.777779,-9.333330,0.217908,0.154270,959.8,41.561471,-6.791185
22,30.221453,77.320000,0.030388,9.722222,-7.694446,0.283140,0.143841,682.4,39.469066,-11.881790
448,24.900694,36.369999,-0.006679,4.944444,-7.194445,0.146452,0.000000,180.6,38.792530,-7.000000
47,31.193737,73.529999,0.003914,9.555554,-8.499999,0.237550,0.115440,713.9,37.314101,-9.209235
462,24.205088,47.230000,0.024702,9.472222,-7.138888,0.185895,0.113739,1029.5,37.258791,-14.452944
2,19.378746,41.829998,-0.028062,8.611111,-9.638889,0.320445,0.160763,370.3,35.971756,9.322888


BMW

Acceleration checks:
Max acceleration >= 10: 0
Max acceleration >= 8: 0
Min acceleration <= -10: 0
Min acceleration <= -8: 1

Energy target checks:
Energy > 30: 0
Energy > 40: 0
Energy > 50: 0
Energy > 60: 0
Energy > 70: 0

Top 10 energy-consumption observations:


,Mean_Speed_kmh,Max_Speed_kmh,Mean_Acceleration_mps2,Max_Acceleration_mps2,Min_Acceleration_mps2,Mean_Positive_Acceleration_mps2,Idling_Proportion,Duration_s,Energy_Consumption_kWh_100km,Mean_Ambient_Temp_C
33,42.284887,135.12,0.156230,3.11,-2.97,0.377419,0.063303,1611.2,23.301433,7.202611
52,60.485600,142.23,-0.005192,3.25,-9.03,0.218213,0.075182,1113.2,21.783572,5.706174
61,64.422194,148.61,-0.023691,2.82,-2.88,0.205300,0.138603,795.7,21.739404,2.193258
58,58.848103,151.23,0.004732,4.09,-3.26,0.226999,0.191926,968.5,21.225910,4.793832
59,58.093566,119.74,-0.033739,2.90,-2.87,0.179713,0.103811,920.8,21.104824,1.088741
47,51.310532,122.31,0.027875,2.60,-1.80,0.185563,0.131320,1109.4,21.052643,5.050799
60,50.021854,114.69,0.010392,2.80,-2.73,0.184827,0.082870,1096.8,21.015502,4.285551
66,44.355806,106.84,-0.020641,3.65,-3.03,0.186253,0.097439,1417.2,21.002172,-3.271403
44,38.007697,141.05,0.008290,4.46,-3.27,0.234079,0.137628,1822.2,20.925972,2.694477
39,85.118739,137.32,-0.025484,2.99,-2.76,0.175658,0.068107,2023.2,20.891196,4.430231


ARGONNE

Acceleration checks:
Max acceleration >= 10: 0
Max acceleration >= 8: 0
Min acceleration <= -10: 0
Min acceleration <= -8: 0

Energy target checks:
Energy > 30: 1
Energy > 40: 0
Energy > 50: 0
Energy > 60: 0
Energy > 70: 0

Top 10 energy-consumption observations:


,Mean_Speed_kmh,Max_Speed_kmh,Mean_Acceleration_mps2,Max_Acceleration_mps2,Min_Acceleration_mps2,Mean_Positive_Acceleration_mps2,Idling_Proportion,Duration_s,Energy_Consumption_kWh_100km,Mean_Test_Cell_Temp_C
1,65.573020,130.531958,-2.273323e-16,1.510991,-1.730040,0.257686,0.038538,549.999,37.690272,23.434814
2,72.226525,133.272664,-1.095385e-06,4.112758,-2.194961,0.695104,0.137421,811.999,26.538541,24.771481
17,39.807207,129.117348,-4.001269e-07,3.303617,-3.545018,0.383460,0.269870,4829.995,25.256664,-16.107683
18,37.127759,129.588885,-1.025976e-06,3.317029,-3.665719,0.403322,0.499515,1443.499,20.946449,-14.301371
15,39.851558,129.670961,-1.215339e-07,3.464551,-3.455611,0.401762,0.268773,4829.995,20.760446,-5.972231
14,4.093332,27.917221,1.562184e-17,1.452876,-3.281265,0.606233,0.792473,92.900,19.260553,0.818833
16,41.062159,129.199425,-8.558486e-08,3.267854,-3.384084,0.415890,0.307495,3112.797,18.353847,-6.131363
12,39.849533,129.814192,-1.131200e-07,3.290206,-3.549489,0.443787,0.269580,4829.995,17.695855,2.364097
7,95.699948,106.913284,7.815831e-17,1.886504,-1.966971,0.192193,0.040856,509.000,15.869896,27.101223
11,92.364641,108.070400,6.652246e-09,2.615178,-2.901282,0.164885,0.088539,678.699,15.813266,39.108745


# Create Harmonised Environmental Temperature Feature

In [18]:
# ============================================================
# Create Harmonised Environmental Temperature Feature
# ============================================================

# Work on copies so original datasets remain untouched
ved_temp = ved.copy()
bmw_temp = bmw.copy()
argonne_temp = argonne.copy()

# Harmonise temperature naming
ved_temp["Environmental_Temperature_C"] = (
    ved_temp["Mean_Temp_C"]
)

bmw_temp["Environmental_Temperature_C"] = (
    bmw_temp["Mean_Ambient_Temp_C"]
)

argonne_temp["Environmental_Temperature_C"] = (
    argonne_temp["Mean_Test_Cell_Temp_C"]
)

print("=" * 70)
print("HARMONISED TEMPERATURE CREATED")
print("=" * 70)

for name, df in {
    "VED": ved_temp,
    "BMW": bmw_temp,
    "ARGONNE": argonne_temp
}.items():

    print(f"\n{name}")
    print(
        df["Environmental_Temperature_C"]
        .describe()
        .round(3)
    )

    print(
        "Missing:",
        df["Environmental_Temperature_C"].isna().sum()
    )

HARMONISED TEMPERATURE CREATED

VED
count    488.000
mean      12.348
std       11.503
min      -14.453
25%        3.074
50%       11.521
75%       22.200
max       35.529
Name: Environmental_Temperature_C, dtype: float64
Missing: 0

BMW
count    68.000
mean     13.214
std       9.675
min      -3.271
25%       4.925
50%       9.089
75%      21.916
max      31.128
Name: Environmental_Temperature_C, dtype: float64
Missing: 0

ARGONNE
count    19.000
mean     16.046
std      17.967
min     -16.108
25%       1.591
50%      23.435
75%      26.481
max      39.109
Name: Environmental_Temperature_C, dtype: float64
Missing: 0


# Build Temperature-Enhanced Master Dataset

In [19]:
# ============================================================
# Build Temperature-Enhanced Master Dataset
# ============================================================

FEATURES_TEMP = [
    "Mean_Speed_kmh",
    "Max_Speed_kmh",
    "Std_Speed_kmh",
    "Mean_Acceleration_mps2",
    "Max_Acceleration_mps2",
    "Min_Acceleration_mps2",
    "Mean_Positive_Acceleration_mps2",
    "Idling_Proportion",
    "Duration_s",
    "Environmental_Temperature_C"
]

TARGET = "Energy_Consumption_kWh_100km"

# ------------------------------------------------------------
# Prepare each dataset
# ------------------------------------------------------------

ved_master_temp = ved_temp[
    FEATURES_TEMP + [TARGET]
].copy()

ved_master_temp["Dataset"] = "VED"


bmw_master_temp = bmw_temp[
    FEATURES_TEMP + [TARGET]
].copy()

bmw_master_temp["Dataset"] = "BMW"


argonne_master_temp = argonne_temp[
    FEATURES_TEMP + [TARGET]
].copy()

argonne_master_temp["Dataset"] = "ARGONNE"


# ------------------------------------------------------------
# Combine
# ------------------------------------------------------------

master_temp = pd.concat(
    [
        ved_master_temp,
        bmw_master_temp,
        argonne_master_temp
    ],
    ignore_index=True
)


# ------------------------------------------------------------
# Audit
# ------------------------------------------------------------

print("=" * 70)
print("TEMPERATURE-ENHANCED MASTER DATASET")
print("=" * 70)

print("\nShape:")
print(master_temp.shape)

print("\nDataset distribution:")
print(master_temp["Dataset"].value_counts())

print("\nMissing values:")
print(master_temp.isna().sum())

print("\nFeatures:")
for feature in FEATURES_TEMP:
    print("✓", feature)

print("\nTarget:")
print("✓", TARGET)

print("\nTemperature statistics:")
print(
    master_temp["Environmental_Temperature_C"]
    .describe()
    .round(3)
)

print("=" * 70)

TEMPERATURE-ENHANCED MASTER DATASET

Shape:
(575, 12)

Dataset distribution:
Dataset
VED        488
BMW         68
ARGONNE     19
Name: count, dtype: int64

Missing values:
Mean_Speed_kmh                     0
Max_Speed_kmh                      0
Std_Speed_kmh                      0
Mean_Acceleration_mps2             0
Max_Acceleration_mps2              0
Min_Acceleration_mps2              0
Mean_Positive_Acceleration_mps2    0
Idling_Proportion                  0
Duration_s                         0
Environmental_Temperature_C        0
Energy_Consumption_kWh_100km       0
Dataset                            0
dtype: int64

Features:
✓ Mean_Speed_kmh
✓ Max_Speed_kmh
✓ Std_Speed_kmh
✓ Mean_Acceleration_mps2
✓ Max_Acceleration_mps2
✓ Min_Acceleration_mps2
✓ Mean_Positive_Acceleration_mps2
✓ Idling_Proportion
✓ Duration_s
✓ Environmental_Temperature_C

Target:
✓ Energy_Consumption_kWh_100km

Temperature statistics:
count    575.000
mean      12.573
std       11.567
min      -16.108
25%    

# TEMP 1 - Prepare Temperature-Enhanced Experiment

In [20]:
# ============================================================
# TEMP 1 - Prepare Temperature-Enhanced Experiment
# ============================================================

from sklearn.model_selection import train_test_split

X_temp = master_temp[FEATURES_TEMP]
y_temp = master_temp[TARGET]
dataset_temp = master_temp["Dataset"]

X_train_temp, X_test_temp, y_train_temp, y_test_temp, \
train_dataset_temp, test_dataset_temp = train_test_split(
    X_temp,
    y_temp,
    dataset_temp,
    test_size=0.20,
    random_state=42,
    stratify=dataset_temp
)

print("=" * 70)
print("TEMPERATURE-ENHANCED TRAIN / TEST SPLIT")
print("=" * 70)

print("\nTraining:", X_train_temp.shape)
print("Testing :", X_test_temp.shape)

print("\nTraining dataset distribution:")
print(train_dataset_temp.value_counts())

print("\nTesting dataset distribution:")
print(test_dataset_temp.value_counts())

print("\nTarget means:")
print(f"Train: {y_train_temp.mean():.4f}")
print(f"Test : {y_test_temp.mean():.4f}")

print("\nTemperature means:")
print(
    "Train:",
    round(X_train_temp["Environmental_Temperature_C"].mean(), 3)
)

print(
    "Test :",
    round(X_test_temp["Environmental_Temperature_C"].mean(), 3)
)

print("=" * 70)

TEMPERATURE-ENHANCED TRAIN / TEST SPLIT

Training: (460, 10)
Testing : (115, 10)

Training dataset distribution:
Dataset
VED        391
BMW         54
ARGONNE     15
Name: count, dtype: int64

Testing dataset distribution:
Dataset
VED        97
BMW        14
ARGONNE     4
Name: count, dtype: int64

Target means:
Train: 16.0933
Test : 16.3739

Temperature means:
Train: 12.612
Test : 12.414


# TEMP 2 - Initial Temperature-Enhanced Models
# Linear Regression + Same Frozen RF Configuration

In [21]:
# ============================================================
# TEMP 2 - Initial Temperature-Enhanced Models
# Linear Regression + Same Frozen RF Configuration
# ============================================================

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error,
    r2_score
)
import numpy as np
import pandas as pd


# ------------------------------------------------------------
# Linear Regression
# ------------------------------------------------------------

lr_temp = LinearRegression()

lr_temp.fit(
    X_train_temp,
    y_train_temp
)

pred_lr_temp = lr_temp.predict(X_test_temp)


# ------------------------------------------------------------
# Random Forest
# Same hyperparameters as previous frozen RF
# ------------------------------------------------------------

rf_temp = RandomForestRegressor(
    n_estimators=480,
    max_depth=11,
    min_samples_split=5,
    min_samples_leaf=3,
    max_features="log2",
    random_state=42,
    n_jobs=-1
)

rf_temp.fit(
    X_train_temp,
    y_train_temp
)

pred_rf_temp = rf_temp.predict(X_test_temp)


# ------------------------------------------------------------
# Evaluation function
# ------------------------------------------------------------

def evaluate_temp_model(name, y_true, predictions):

    mae = mean_absolute_error(
        y_true,
        predictions
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_true,
            predictions
        )
    )

    r2 = r2_score(
        y_true,
        predictions
    )

    mape = mean_absolute_percentage_error(
        y_true,
        predictions
    ) * 100

    return {
        "Model": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "MAPE_pct": mape
    }


temp_initial_results = pd.DataFrame([
    evaluate_temp_model(
        "Linear Regression + Temperature",
        y_test_temp,
        pred_lr_temp
    ),

    evaluate_temp_model(
        "Random Forest + Temperature",
        y_test_temp,
        pred_rf_temp
    )
])


print("=" * 75)
print("INITIAL TEMPERATURE-ENHANCED RESULTS")
print("=" * 75)

display(
    temp_initial_results.round(4)
)


# Reference values come from the ML notebook; print only if available
if "final_mae_frozen" in globals():
    print("\nPrevious behaviour-only reference:")
    print(
        f"Frozen RF -> "
        f"MAE={final_mae_frozen:.4f}, "
        f"RMSE={final_rmse_frozen:.4f}, "
        f"R²={final_r2_frozen:.4f}, "
        f"MAPE={final_mape_frozen:.2f}%"
    )

INITIAL TEMPERATURE-ENHANCED RESULTS


,Model,MAE,RMSE,R2,MAPE_pct
0,Linear Regression + Temperature,3.7678,5.2809,0.1767,25.8323
1,Random Forest + Temperature,3.0680,4.4235,0.4223,20.6391


In [22]:
# ============================================================
# Behaviour-Only vs Temperature-Enhanced Comparison
# ============================================================

comparison_temperature = pd.DataFrame([
    {
        "Model": "RF - Behaviour Only",
        "MAE": 4.1417,
        "RMSE": 6.1415,
        "R2": -0.1135,
        "MAPE_pct": 27.52
    },
    {
        "Model": "RF + Environmental Temperature",
        "MAE": mean_absolute_error(y_test_temp, pred_rf_temp),
        "RMSE": np.sqrt(mean_squared_error(y_test_temp, pred_rf_temp)),
        "R2": r2_score(y_test_temp, pred_rf_temp),
        "MAPE_pct": mean_absolute_percentage_error(
            y_test_temp,
            pred_rf_temp
        ) * 100
    }
])

display(comparison_temperature.round(4))

,Model,MAE,RMSE,R2,MAPE_pct
0,RF - Behaviour Only,4.1417,6.1415,-0.1135,27.5200
1,RF + Environmental Temperature,3.0680,4.4235,0.4223,20.6391


# TEMP 3 - Training-Only Cross-Validation

In [23]:
# ============================================================
# TEMP 3 - Training-Only Cross-Validation
# ============================================================

from sklearn.model_selection import KFold, cross_val_score

TEMP_CV = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

temp_cv_scores = cross_val_score(
    rf_temp,
    X_train_temp,
    y_train_temp,
    cv=TEMP_CV,
    scoring="r2",
    n_jobs=-1
)

print("=" * 70)
print("TEMPERATURE-ENHANCED RF — TRAINING-ONLY CV")
print("=" * 70)

for i, score in enumerate(temp_cv_scores, start=1):
    print(f"Fold {i}: R² = {score:.4f}")

print()
print(f"Mean CV R² : {temp_cv_scores.mean():.4f}")
print(f"CV SD      : {temp_cv_scores.std():.4f}")

print("=" * 70)

TEMPERATURE-ENHANCED RF — TRAINING-ONLY CV
Fold 1: R² = 0.5716
Fold 2: R² = 0.5367
Fold 3: R² = 0.5156
Fold 4: R² = 0.5351
Fold 5: R² = 0.4702

Mean CV R² : 0.5259
CV SD      : 0.0332


# TEMP 4 - Optuna Random Forest Tuning
# TRAINING DATA ONLY

In [24]:
# ============================================================
# TEMP 4 - Optuna Random Forest Tuning
# TRAINING DATA ONLY
# ============================================================

import optuna
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score


def objective_rf_temp(trial):

    model = RandomForestRegressor(

        n_estimators=trial.suggest_int(
            "n_estimators",
            150,
            600
        ),

        max_depth=trial.suggest_int(
            "max_depth",
            3,
            15
        ),

        min_samples_split=trial.suggest_int(
            "min_samples_split",
            2,
            15
        ),

        min_samples_leaf=trial.suggest_int(
            "min_samples_leaf",
            1,
            10
        ),

        max_features=trial.suggest_categorical(
            "max_features",
            ["sqrt", "log2", 0.7, 1.0]
        ),

        random_state=42,
        n_jobs=-1
    )

    scores = cross_val_score(
        model,
        X_train_temp,
        y_train_temp,
        cv=TEMP_CV,
        scoring="r2",
        n_jobs=-1
    )

    return scores.mean()


study_rf_temp = optuna.create_study(
    direction="maximize"
)

study_rf_temp.optimize(
    objective_rf_temp,
    n_trials=30,
    show_progress_bar=True
)


print("=" * 70)
print("TEMPERATURE RF — OPTUNA RESULTS")
print("=" * 70)

print(f"\nBest CV R²: {study_rf_temp.best_value:.4f}")

print("\nBest parameters:")
for parameter, value in study_rf_temp.best_params.items():
    print(f"{parameter}: {value}")

print("=" * 70)

[I 2026-10-18 23:00:17,573] A new study created in memory with name: no-name-fd6693fe-c7e8-45dd-acdf-ef58f794edf2


  0%|          | 0/30 [00:00<?, ?it/s]

[I 2026-10-18 23:00:21,485] Trial 0 finished with value: 0.5114465689485417 and parameters: {'n_estimators': 295, 'max_depth': 6, 'min_samples_split': 7, 'min_samples_leaf': 3, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.5114465689485417.
[I 2026-10-18 23:00:22,592] Trial 1 finished with value: 0.5361890692545392 and parameters: {'n_estimators': 365, 'max_depth': 9, 'min_samples_split': 5, 'min_samples_leaf': 6, 'max_features': 0.7}. Best is trial 1 with value: 0.5361890692545392.
[I 2026-10-18 23:00:23,049] Trial 2 finished with value: 0.5471646434800246 and parameters: {'n_estimators': 152, 'max_depth': 5, 'min_samples_split': 6, 'min_samples_leaf': 4, 'max_features': 0.7}. Best is trial 2 with value: 0.5471646434800246.
[I 2026-10-18 23:00:24,709] Trial 3 finished with value: 0.5295950310426937 and parameters: {'n_estimators': 557, 'max_depth': 11, 'min_samples_split': 4, 'min_samples_leaf': 7, 'max_features': 1.0}. Best is trial 2 with value: 0.5471646434800246.
[I 2026-

# TEMP 4.1 - Inspect Best Optuna Parameters

In [25]:
# ============================================================
# TEMP 4.1 - Inspect Best Optuna Parameters
# ============================================================

print("=" * 70)
print("BEST TEMPERATURE RF CONFIGURATION")
print("=" * 70)

print(f"\nBest CV R²: {study_rf_temp.best_value:.6f}")

print("\nBest parameters:")
for parameter, value in study_rf_temp.best_params.items():
    print(f"{parameter}: {value}")

print("=" * 70)

BEST TEMPERATURE RF CONFIGURATION

Best CV R²: 0.556194

Best parameters:
n_estimators: 269
max_depth: 13
min_samples_split: 10
min_samples_leaf: 4
max_features: 1.0


In [26]:
print(study_rf_temp.best_params)

{'n_estimators': 269, 'max_depth': 13, 'min_samples_split': 10, 'min_samples_leaf': 4, 'max_features': 1.0}


# TEMP 5 - AUTHORITATIVE FINAL TEMPERATURE-ENHANCED RF
# Fit on training set -> Evaluate on untouched test set

In [27]:
# ============================================================
# TEMP 5 - AUTHORITATIVE FINAL TEMPERATURE-ENHANCED RF
# Fit on training set -> Evaluate on untouched test set
# ============================================================

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error,
    r2_score
)
import numpy as np


# ------------------------------------------------------------
# 1. Freeze Optuna-selected configuration
# ------------------------------------------------------------

final_rf_temp = RandomForestRegressor(
    n_estimators=350,
    max_depth=13,
    min_samples_split=8,
    min_samples_leaf=3,
    max_features=0.7,
    random_state=42,
    n_jobs=-1
)


# ------------------------------------------------------------
# 2. Fit ONLY on training data
# ------------------------------------------------------------

final_rf_temp.fit(
    X_train_temp,
    y_train_temp
)


# ------------------------------------------------------------
# 3. Predict untouched test set
# ------------------------------------------------------------

final_pred_temp = final_rf_temp.predict(
    X_test_temp
)


# ------------------------------------------------------------
# 4. Authoritative test metrics
# ------------------------------------------------------------

final_temp_mae = mean_absolute_error(
    y_test_temp,
    final_pred_temp
)

final_temp_rmse = np.sqrt(
    mean_squared_error(
        y_test_temp,
        final_pred_temp
    )
)

final_temp_r2 = r2_score(
    y_test_temp,
    final_pred_temp
)

final_temp_mape = mean_absolute_percentage_error(
    y_test_temp,
    final_pred_temp
) * 100


# ------------------------------------------------------------
# 5. Report
# ------------------------------------------------------------

print("=" * 75)
print("AUTHORITATIVE TEMPERATURE-ENHANCED RANDOM FOREST")
print("UNTOUCHED TEST SET")
print("=" * 75)

print(f"\nMAE  : {final_temp_mae:.4f} kWh/100 km")
print(f"RMSE : {final_temp_rmse:.4f} kWh/100 km")
print(f"R²   : {final_temp_r2:.4f}")
print(f"MAPE : {final_temp_mape:.2f}%")

print(f"\nTest observations: {len(y_test_temp)}")

print("\nTraining-only tuning result:")
print(f"Best CV R²: {study_rf_temp.best_value:.4f}")

print("=" * 75)

AUTHORITATIVE TEMPERATURE-ENHANCED RANDOM FOREST
UNTOUCHED TEST SET

MAE  : 3.1081 kWh/100 km
RMSE : 4.6104 kWh/100 km
R²   : 0.3725
MAPE : 20.49%

Test observations: 115

Training-only tuning result:
Best CV R²: 0.5562


# TEMP 6 - Leave-One-Dataset-Out Validation
# Temperature-Enhanced Final RF

In [28]:
# ============================================================
# TEMP 6 - Leave-One-Dataset-Out Validation
# Temperature-Enhanced Final RF
# ============================================================

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error,
    r2_score
)

import numpy as np
import pandas as pd


cross_dataset_temp_rows = []

datasets_unique = [
    "VED",
    "BMW",
    "ARGONNE"
]


for held_out_dataset in datasets_unique:

    print("=" * 75)
    print(f"HELD-OUT DATASET: {held_out_dataset}")
    print("=" * 75)

    # --------------------------------------------------------
    # Split by complete dataset
    # --------------------------------------------------------

    train_mask = (
        master_temp["Dataset"] != held_out_dataset
    )

    test_mask = (
        master_temp["Dataset"] == held_out_dataset
    )

    X_train_cross = master_temp.loc[
        train_mask,
        FEATURES_TEMP
    ]

    y_train_cross = master_temp.loc[
        train_mask,
        TARGET
    ]

    X_test_cross = master_temp.loc[
        test_mask,
        FEATURES_TEMP
    ]

    y_test_cross = master_temp.loc[
        test_mask,
        TARGET
    ]


    # --------------------------------------------------------
    # Use frozen Optuna-selected RF configuration
    # --------------------------------------------------------

    cross_model = RandomForestRegressor(
        n_estimators=350,
        max_depth=13,
        min_samples_split=8,
        min_samples_leaf=3,
        max_features=0.7,
        random_state=42,
        n_jobs=-1
    )


    # --------------------------------------------------------
    # Train on the OTHER datasets only
    # --------------------------------------------------------

    cross_model.fit(
        X_train_cross,
        y_train_cross
    )


    # --------------------------------------------------------
    # Predict completely unseen dataset
    # --------------------------------------------------------

    cross_pred = cross_model.predict(
        X_test_cross
    )


    # --------------------------------------------------------
    # Metrics
    # --------------------------------------------------------

    mae = mean_absolute_error(
        y_test_cross,
        cross_pred
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_test_cross,
            cross_pred
        )
    )

    r2 = r2_score(
        y_test_cross,
        cross_pred
    )

    mape = mean_absolute_percentage_error(
        y_test_cross,
        cross_pred
    ) * 100


    cross_dataset_temp_rows.append({
        "Held_Out_Dataset": held_out_dataset,
        "Train_N": len(X_train_cross),
        "Test_N": len(X_test_cross),
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "MAPE_pct": mape
    })


    print(f"Training observations : {len(X_train_cross)}")
    print(f"Testing observations  : {len(X_test_cross)}")

    print(f"\nMAE  : {mae:.4f}")
    print(f"RMSE : {rmse:.4f}")
    print(f"R²   : {r2:.4f}")
    print(f"MAPE : {mape:.2f}%")

    print()


# ============================================================
# Final table
# ============================================================

cross_dataset_temp_results = pd.DataFrame(
    cross_dataset_temp_rows
)

print("=" * 75)
print("TEMPERATURE-ENHANCED CROSS-DATASET VALIDATION")
print("=" * 75)

display(
    cross_dataset_temp_results.round(4)
)

HELD-OUT DATASET: VED
Training observations : 87
Testing observations  : 488

MAE  : 4.6752
RMSE : 6.8855
R²   : 0.1507
MAPE : 36.71%

HELD-OUT DATASET: BMW
Training observations : 507
Testing observations  : 68

MAE  : 2.6906
RMSE : 3.2389
R²   : 0.1107
MAPE : 16.49%

HELD-OUT DATASET: ARGONNE
Training observations : 556
Testing observations  : 19

MAE  : 6.4130
RMSE : 8.7491
R²   : -0.5739
MAPE : 46.83%

TEMPERATURE-ENHANCED CROSS-DATASET VALIDATION


,Held_Out_Dataset,Train_N,Test_N,MAE,RMSE,R2,MAPE_pct
0,VED,87,488,4.6752,6.8855,0.1507,36.7121
1,BMW,507,68,2.6906,3.2389,0.1107,16.4892
2,ARGONNE,556,19,6.4130,8.7491,-0.5739,46.8321


# Save Temperature-Enhanced Master Dataset

In [29]:
# ============================================================
# Save Temperature-Enhanced Master Dataset
# ============================================================

from pathlib import Path

OUTPUT_PATH = Path(
    r"C:\Projects\DS7010_Dissertation\01-Datasets\Harmonised"
)

OUTPUT_PATH.mkdir(
    parents=True,
    exist_ok=True
)

MASTER_TEMP_PATH = (
    OUTPUT_PATH /
    "Master_EV_Dataset_With_Temperature.csv"
)

master_temp.to_csv(
    MASTER_TEMP_PATH,
    index=False
)

print("=" * 70)
print("TEMPERATURE-ENHANCED MASTER DATASET SAVED")
print("=" * 70)

print("\nPath:")
print(MASTER_TEMP_PATH)

print("\nShape:")
print(master_temp.shape)

print("\nDataset distribution:")
print(master_temp["Dataset"].value_counts())

print("\nMissing values:")
print(master_temp.isna().sum().sum())

print("=" * 70)

TEMPERATURE-ENHANCED MASTER DATASET SAVED

Path:
C:\Projects\DS7010_Dissertation\01-Datasets\Harmonised\Master_EV_Dataset_With_Temperature.csv

Shape:
(575, 12)

Dataset distribution:
Dataset
VED        488
BMW         68
ARGONNE     19
Name: count, dtype: int64

Missing values:
0
